In [1]:
# 第一部分：导入必要的库
# ============================================

import numpy as np
import umap

from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC

import warnings
warnings.filterwarnings('ignore')


c:\Users\Administrator\.conda\envs\quantum\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# 第二部分：可调参数配置区（UMAP降维 + 线性SVM 经典对照组）
# ============================================

# 数据相关参数（与量子实验保持一致）
SAMPLE_RATIO = 1            # 数据采样比例
TEST_SIZE = 0.2             # 测试集比例
PCA_COMPONENTS = 8          # 降维目标维度（与量子比特数一致）

# 降维方法
DIM_REDUCTION_METHOD = 'umap'

# UMAP 超参数（n_neighbors=10 / min_dist=0.5）
UMAP_N_NEIGHBORS = 10       # 局部邻域大小
UMAP_MIN_DIST = 0.5         # 最小距离
UMAP_METRIC = 'euclidean'   # 距离度量

# SVM 参数
SVM_KERNEL = 'linear'       # 核函数
SVM_C = 1.0                 # 正则化参数（默认值，不调优）

# 随机种子（与量子实验一致）
SEED = 42

# 初始化设置
# ============================================
np.random.seed(SEED)


In [3]:
# 第三部分：数据加载、UMAP降维与预处理（与实验三基线一致）
# ============================================

import os
import gzip
import struct
import urllib.request

from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split

_MNIST_FILES = {
    'train_images': ('train-images-idx3-ubyte.gz', 'http://yann.lecun.com/exdb/mnist/train-images-idx3-ubyte.gz'),
    'train_labels': ('train-labels-idx1-ubyte.gz', 'http://yann.lecun.com/exdb/mnist/train-labels-idx1-ubyte.gz'),
    'test_images':  ('t10k-images-idx3-ubyte.gz',  'http://yann.lecun.com/exdb/mnist/t10k-images-idx3-ubyte.gz'),
    'test_labels':  ('t10k-labels-idx1-ubyte.gz',  'http://yann.lecun.com/exdb/mnist/t10k-labels-idx1-ubyte.gz'),
}

def _ensure_mnist_files(data_dir='./MNIST'):
    """下载/校验 MNIST 原始 IDX 文件（有本地缓存则不重复下载）"""
    os.makedirs(data_dir, exist_ok=True)
    paths = {}
    for key, (fname, url) in _MNIST_FILES.items():
        path = os.path.join(data_dir, fname)
        if not os.path.exists(path):
            print(f"下载 {fname} ...")
            try:
                urllib.request.urlretrieve(url, path)
            except Exception as e:
                raise RuntimeError(
                    f"下载 {fname} 失败：{e}。请确认本机可联网，"
                    f"或把原始电脑上的 ./MNIST 目录（含 4 个 .gz 文件）拷贝到本机后重试。"
                )
        paths[key] = path
    return paths

def _read_idx_images(path):
    with gzip.open(path, 'rb') as f:
        magic, n, rows, cols = struct.unpack('>IIII', f.read(16))
        return np.frombuffer(f.read(), dtype=np.uint8).reshape(n, rows, cols)

def _read_idx_labels(path):
    with gzip.open(path, 'rb') as f:
        magic, n = struct.unpack('>II', f.read(8))
        return np.frombuffer(f.read(), dtype=np.uint8)

def load_mnist_data():
    """加载MNIST数据集（不依赖 torchvision，直接解析官方 IDX 文件，数据与 torchvision 一致）"""
    print("正在加载MNIST数据集...")
    paths = _ensure_mnist_files('./MNIST')

    train_images = _read_idx_images(paths['train_images'])   # (60000, 28, 28)
    train_labels = _read_idx_labels(paths['train_labels'])   # (60000,)
    test_images  = _read_idx_images(paths['test_images'])    # (10000, 28, 28)
    test_labels  = _read_idx_labels(paths['test_labels'])    # (10000,)

    def flatten(imgs):
        return imgs.astype(np.float32).reshape(len(imgs), -1) / 255.0

    # 合并（顺序与 torchvision 一致：先 60000 训练 + 后 10000 测试）
    all_images = np.concatenate([flatten(train_images), flatten(test_images)])
    all_labels = np.concatenate([train_labels, test_labels]).astype(np.int64)

    sampled_indices = np.random.choice(
        len(all_images),
        int(len(all_images) * SAMPLE_RATIO),
        replace=False
    )
    sampled_images = all_images[sampled_indices]
    sampled_labels = all_labels[sampled_indices]

    print(f"总样本数: {len(all_images)}, 采样后: {len(sampled_images)}")
    print(f"类别分布: {np.unique(sampled_labels, return_counts=True)}")

    return sampled_images, sampled_labels

def preprocess_data_umap(images, umap_model=None, final_scaler=None, is_train=True):
    """
    UMAP降维 + MinMax(-1,1) 预处理函数（与实验三基线一致）
    训练集 fit，测试集 transform；返回 numpy 特征 (N, 8)
    """
    if is_train:
        # 先缩放到[0,1]用于UMAP
        temp_scaler = MinMaxScaler(feature_range=(0, 1))
        scaled_for_umap = temp_scaler.fit_transform(images)

        # 训练UMAP
        umap_model = umap.UMAP(
            n_components=PCA_COMPONENTS,
            n_neighbors=UMAP_N_NEIGHBORS,
            min_dist=UMAP_MIN_DIST,
            metric=UMAP_METRIC,
            random_state=SEED
        )
        reduced_data = umap_model.fit_transform(scaled_for_umap)
        print(f"UMAP降维完成，输出形状: {reduced_data.shape}")

        # 将UMAP输出归一化到[-1,1]作为最终特征
        final_scaler = MinMaxScaler(feature_range=(-1, 1))
        features = final_scaler.fit_transform(reduced_data)

        # 将temp_scaler保存到umap_model中，供测试时使用
        umap_model.temp_scaler = temp_scaler
    else:
        # 测试集：使用训练好的 temp_scaler 和 final_scaler
        temp_scaler = umap_model.temp_scaler
        scaled_for_umap = temp_scaler.transform(images)
        reduced_data = umap_model.transform(scaled_for_umap)
        features = final_scaler.transform(reduced_data)

    return features, final_scaler, umap_model


In [4]:
# 第四部分：SVM 训练与评估函数
# ============================================

def train_evaluate_svm(train_features, train_labels, test_features, test_labels,
                       kernel='linear', C=1.0):
    """训练SVM，返回 (模型, 训练集准确率, 测试集准确率)"""
    print(f"\n训练 SVM (kernel={kernel}, C={C}) ...")
    model = SVC(kernel=kernel, C=C, random_state=SEED)
    model.fit(train_features, train_labels)

    train_pred = model.predict(train_features)
    test_pred = model.predict(test_features)

    train_acc = (train_pred == train_labels).mean()
    test_acc = (test_pred == test_labels).mean()

    print(f"  训练集准确率: {train_acc:.4f}")
    print(f"  测试集准确率: {test_acc:.4f}")

    return model, train_acc, test_acc


In [5]:
# 第五部分：主程序 - UMAP降维 + 线性SVM 经典对照组
# ============================================

if __name__ == "__main__":
    print("=" * 60)
    print("UMAP降维 + 线性SVM 经典对照组")
    print("=" * 60)

    # 1. 加载数据
    print("\n步骤1：加载MNIST数据")
    all_images, all_labels = load_mnist_data()
    print(f"数据大小: {len(all_images)}")

    # 2. 划分数据集（与量子实验一致）
    print("\n步骤2：划分训练集和测试集")
    train_images, test_images, train_labels, test_labels = train_test_split(
        all_images, all_labels, test_size=TEST_SIZE, random_state=SEED, stratify=all_labels
    )
    print(f"训练集: {len(train_images)} 样本, 测试集: {len(test_images)} 样本")

    # 3. UMAP降维 + 归一化
    print("\n步骤3：UMAP降维预处理")
    train_features, final_scaler, umap_model = preprocess_data_umap(
        train_images, is_train=True
    )
    test_features, _, _ = preprocess_data_umap(
        test_images, umap_model=umap_model, final_scaler=final_scaler, is_train=False
    )
    print(f"特征形状 - 训练集: {train_features.shape}, 测试集: {test_features.shape}")

    # 4. 训练并评估 SVM
    print("\n步骤4：训练并评估线性SVM")
    model, train_acc, test_acc = train_evaluate_svm(
        train_features, train_labels, test_features, test_labels,
        kernel=SVM_KERNEL, C=SVM_C
    )

    # 5. 结果汇总
    print("\n" + "=" * 60)
    print("UMAP + 线性SVM 对照组结果汇总")
    print("=" * 60)
    print(f"核函数: {SVM_KERNEL}, C: {SVM_C}")
    print(f"训练集准确率: {train_acc:.4f}")
    print(f"测试集准确率: {test_acc:.4f}")

    # 6. 保存结果（带 UMAP 参数后缀）
    np.savez('umap_svm_control_nn10_md0.5_results.npz',
             kernel=SVM_KERNEL,
             C=SVM_C,
             train_acc=train_acc,
             test_acc=test_acc)
    print("\n结果已保存到 umap_svm_control_nn10_md0.5_results.npz")


UMAP降维 + 线性SVM 经典对照组

步骤1：加载MNIST数据
正在加载MNIST数据集...
总样本数: 70000, 采样后: 70000
类别分布: (array([0, 1, 2, 3, 4, 5, 6, 7, 8, 9], dtype=int64), array([6903, 7877, 6990, 7141, 6824, 6313, 6876, 7293, 6825, 6958],
      dtype=int64))
数据大小: 70000

步骤2：划分训练集和测试集
训练集: 56000 样本, 测试集: 14000 样本

步骤3：UMAP降维预处理
UMAP降维完成，输出形状: (56000, 8)
特征形状 - 训练集: (56000, 8), 测试集: (14000, 8)

步骤4：训练并评估线性SVM

训练 SVM (kernel=linear, C=1.0) ...
  训练集准确率: 0.9650
  测试集准确率: 0.9450

UMAP + 线性SVM 对照组结果汇总
核函数: linear, C: 1.0
训练集准确率: 0.9650
测试集准确率: 0.9450

结果已保存到 umap_svm_control_nn10_md0.5_results.npz
